Transformer_Decoder架构

In [1]:
import math
import torch
import torch.nn as nn

import warnings
warnings.filterwarnings(action="ignore")

In [ ]:
class SimpleDecoder(nn.Module):
    def __init__(self,hidden_dim,num_heads,dropout=0.1):
        super().__init__()

        self.num_heads = num_heads
        self.hidden_dim = hidden_dim
        self.head_dim = hidden_dim // num_heads
        self.dropout = dropout

        self.q = nn.Linear(hidden_dim,hidden_dim)
        self.k = nn.Linear(hidden_dim,hidden_dim)
        self.v = nn.Linear(hidden_dim,hidden_dim)
        self.output_proj = nn.Linear(hidden_dim,hidden_dim)
        self.drop_attn = nn.Dropout(self.dropout)
        self.LayerNorm_attn = nn.LayerNorm(hidden_dim, eps=1e-5)

        # 前馈层FFN
        self.upward_proj = nn.Linear(hidden_dim,hidden_dim*4)
        self.downward_proj = nn.Linear(hidden_dim*4,hidden_dim)
        # eps: 防止分母为0
        self.LayerNorm_ffn = nn.LayerNorm(hidden_dim,eps=1e-5)
        self.act_fn = nn.GELU()
        self.drop_ffn = nn.Dropout(self.dropout)
       
    def attention_output(self,query,key,value,attention_mask=None):
        # 1 计算attention_weight
        attention_value = query @ key.transpose(-2,-1)
        attention_weights = attention_value / math.sqrt(self.head_dim) #[3,8,4,16] @ [3,8,16,4] -> [3,8,4,4]
        #print(attention_weights.shape)

        # 2.掩码
        if attention_mask is not None:
            # padding mask + 下三角casual mask
            attention_mask = attention_mask.tril()
            attention_weights = attention_weights.masked_fill(attention_mask == 0, float("-1e20"))
        else: # casual mask
            attention_mask = torch.ones_like(attention_weights).tril()
            attention_weights = attention_weights.masked_fill(attention_mask == 0, float("-1e20")) 

        # 3.计算softmax
        attention_weights = torch.softmax(attention_weights,dim=-1)
        #print(attention_weights.shape)
        
        # 4.计算dropout
        attention_weights = self.drop_attn(attention_weights)
        #print(attention_weights.shape)
        
        #5. 计算output并reshape输入
        output_mid = attention_weights @ value #[3,8,4,8]
        #print(output_mid.shape)
        output_mid = output_mid.transpose(1,2).contiguous()
        #print(output_mid.shape) # [3,4,8,8]
        batch,seq_len,_,_= output_mid.size()
        output_mid = output_mid.view(batch,seq_len,-1)
        #print(output_mid.shape) # [3,4,64]
        output = self.output_proj(output_mid)

        return output


    def attention_block(self, x, attention_mask=None):
        batch_size,seq_len,_ = x.shape

        query = self.q(x).view(batch_size,seq_len,self.num_heads,self.head_dim).transpose(1,2)
        key = self.k(x).view(batch_size,seq_len,self.num_heads,self.head_dim).transpose(1,2)
        value = self.v(x).view(batch_size,seq_len,self.num_heads,self.head_dim).transpose(1,2)

        output = self.attention_output(
            query,
            key,
            value,
            attention_mask=attention_mask,
        )
        return self.LayerNorm_attn(x + output)
        

    def ffn_block(self,x):
        up = self.upward_proj(x) #[3,4,256]
        #print(up.shape)
        up = self.act_fn(up)
        #print(up.shape)
        down = self.downward_proj(up) # [3,4,64]
        #print(down.shape)
        down = self.drop_ffn(down)
        #print(down.shape)

        # 先残差后LayerNorm - 标准写法
        # 先LayerNorm后残差 - GPT2
        return self.LayerNorm_ffn(x+down)

    def forward(self, X, attention_mask=None):
        # X 一般假设是已经经过 embedding 的输入， (batch, seq, hidden_dim)
        # attention_mask 一般指的是 tokenizer 后返回的 mask 结果，表示哪些样本需要忽略
        # shape 一般是： (batch, nums_head, seq)

        att_output = self.attention_block(X, attention_mask=attention_mask)
        ffn_output = self.ffn_block(att_output)
        return ffn_output
    

In [28]:
class Decoder(nn.Module):
    def __init__(self,):
        super().__init__()
        self.layer_list = nn.ModuleList(
            [
                SimpleDecoder(64,8) for i in range(5)
            ]
        )
        self.emb = nn.Embedding(12,64)
        self.out = nn.Linear(64,12)
    
    def forward(self,X,mask=None):
        x = self.emb(X)
        for i, l in enumerate(self.layer_list):
            x = l(x,mask)
        print(x.shape)
        print('---------')
        output = self.out(x)
        return torch.softmax(output,dim=-1)
        

In [29]:
x = torch.randint(0, 12, (3, 4))
net = Decoder()

# x = torch.rand(3, 4, 64)
# net = SimpleDecoder(64, 8,0.1)
mask = (
    torch.tensor([[1, 1, 1, 1], [1, 1, 0, 0], [1, 1, 1, 0]])
    .unsqueeze(1)
    .unsqueeze(2)
    .repeat(1, 8, 4, 1)
)
print(mask.shape)
net(x, mask).shape

torch.Size([3, 8, 4, 4])
torch.Size([3, 4, 64])
---------


torch.Size([3, 4, 12])